# Convert Allen V1 (core_nll_7) to libsonata-compatible circuit

The original `core_nll_7` BMTK config relies on `${configdir}` manifest
tokens and on per-`node_type_id` `dynamics_params` JSON files under
`components/cell_models/`, which libsonata/bluepysnap cannot parse.

This notebook rewrites the V1 / LGN / BKG node populations into self-contained
SONATA HDF5 files that mirror the `fly_nodes.h5` layout
(`../../../obi-output/circuit/nest_fly_brain/fly_nodes.h5`) by embedding
`model_template` and `dynamics_params` directly into each node group.

### Neuron model

All 201 V1 cell types in `core_nll_7` use the Allen GLIF LIF-ASC model
(`nest:glif_psc_double_alpha`), loaded by NEST via the
`glif_psc_double_alpha_module` (see `biorealistic-v1-model/run_pointnet.py`).
Each cell type has its own `<specimen_id>_glif_lif_asc_config.json` with:

| Group         | Parameters                                              | Shape       |
|---------------|---------------------------------------------------------|-------------|
| Passive       | `V_m`, `V_th`, `g`, `E_L`, `C_m`, `t_ref`, `V_reset`    | scalar      |
| After-spike   | `asc_init`, `asc_decay`, `asc_amps`                     | length 2    |
| Synapse kinetics (10 receptor types) | `tau_syn_fast`, `tau_syn_slow`, `amp_slow` | length 10 |
| Mode flags    | `spike_dependent_threshold=False`, `after_spike_currents=True`, `adapting_threshold=False` | scalar, constant across types |

For each V1 neuron, the parameter values are looked up from its `node_type_id`
and stored as per-cell datasets under `nodes/v1/0/dynamics_params/`. Vector
parameters become 2-D datasets of shape `(n_cells, k)`. `model_template` is a
scalar string (`glif_psc_double_alpha`) so it can be passed directly to
`nest.Create()`.

LGN and BKG are virtual populations (spike generators) and keep their node
metadata but have no `model_template` / `dynamics_params`.

**Output:** `../../../obi-output/circuit/nest_core_nll_7/`

In [1]:
import json
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

CIRCUIT_BASE = Path("/Users/james/Documents/obi/code/obi-main/core_nll_7")
NETWORK_DIR = CIRCUIT_BASE / "network"
CELL_MODELS_DIR = CIRCUIT_BASE / "components" / "cell_models"
OUTPUT_DIR = Path("../../../obi-output/circuit/nest_core_nll_7")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NEST_MODEL = "glif_psc_double_alpha"
SCALAR_PARAMS = ["V_m", "V_th", "g", "E_L", "C_m", "t_ref", "V_reset"]
VECTOR_PARAMS = {
    "asc_init": 2,
    "asc_decay": 2,
    "asc_amps": 2,
    "tau_syn_fast": 10,
    "tau_syn_slow": 10,
    "amp_slow": 10,
}
FLAG_PARAMS = ["spike_dependent_threshold", "after_spike_currents", "adapting_threshold"]

### Build per-cell dynamics_params from the node_types CSV + JSON configs

Read `v1_node_types.csv` to map each neuron's `node_type_id` -> `dynamics_params`
JSON filename, load every unique JSON from `components/cell_models/`, and
assemble per-cell arrays grouped by scalar / vector / flag fields.

In [2]:
with h5py.File(NETWORK_DIR / "v1_nodes.h5", "r") as src:
    v1_node_id = src["nodes/v1/node_id"][:]
    v1_node_type_id = src["nodes/v1/node_type_id"][:]
    v1_x = src["nodes/v1/0/x"][:]
    v1_y = src["nodes/v1/0/y"][:]
    v1_z = src["nodes/v1/0/z"][:]
    v1_tuning_angle = src["nodes/v1/0/tuning_angle"][:]
    v1_target_sizes = src["nodes/v1/0/target_sizes"][:]

n_v1 = v1_node_id.size
print(f"V1 neurons: {n_v1:,}")

v1_types_df = pd.read_csv(NETWORK_DIR / "v1_node_types.csv", sep=" ")
type_to_json = dict(zip(v1_types_df["node_type_id"], v1_types_df["dynamics_params"]))

json_cache: dict[str, dict] = {}
for fn in set(type_to_json.values()):
    with (CELL_MODELS_DIR / fn).open() as f:
        json_cache[fn] = json.load(f)
print(f"Loaded {len(json_cache)} unique dynamics_params JSON files for {len(type_to_json)} node_types")

scalars = {p: np.empty(n_v1, dtype=np.float64) for p in SCALAR_PARAMS}
vectors = {p: np.empty((n_v1, k), dtype=np.float64) for p, k in VECTOR_PARAMS.items()}

for tid in np.unique(v1_node_type_id):
    mask = v1_node_type_id == tid
    params = json_cache[type_to_json[int(tid)]]
    for p in SCALAR_PARAMS:
        scalars[p][mask] = params[p]
    for p in VECTOR_PARAMS:
        vectors[p][mask] = params[p]

flag_values: dict[str, bool] = {}
for p in FLAG_PARAMS:
    vals = {json_cache[fn][p] for fn in json_cache}
    if len(vals) != 1:
        msg = f"{p} is not constant across V1 node_types: {vals}"
        raise ValueError(msg)
    flag_values[p] = bool(next(iter(vals)))
print(f"Constant flags across all 201 node_types: {flag_values}")

V1 neurons: 66,952
Loaded 201 unique dynamics_params JSON files for 201 node_types
Constant flags across all 201 node_types: {'spike_dependent_threshold': False, 'after_spike_currents': True, 'adapting_threshold': False}


### Write `v1_nodes.h5`

Mirrors the `fly_nodes.h5` layout: one group `0` holding the node metadata,
a scalar `model_template` string, and a `dynamics_params/` subgroup whose
datasets are either scalars (flags that are constant across the population)
or per-cell arrays (scalar or vector parameters).

In [3]:
v1_nodes_path = OUTPUT_DIR / "v1_nodes.h5"

with h5py.File(v1_nodes_path, "w") as f:
    pop = f.create_group("nodes/v1")
    pop.create_dataset("node_id", data=v1_node_id.astype(np.uint64))
    pop.create_dataset("node_type_id", data=v1_node_type_id.astype(np.uint64))
    pop.create_dataset("node_group_id", data=np.zeros(n_v1, dtype=np.uint64))
    pop.create_dataset("node_group_index", data=np.arange(n_v1, dtype=np.uint64))

    grp = pop.create_group("0")
    grp.create_dataset("x", data=v1_x)
    grp.create_dataset("y", data=v1_y)
    grp.create_dataset("z", data=v1_z)
    grp.create_dataset("tuning_angle", data=v1_tuning_angle)
    grp.create_dataset("target_sizes", data=v1_target_sizes)
    grp.create_dataset("model_template", data=NEST_MODEL)

    dp = grp.create_group("dynamics_params")
    for p, arr in scalars.items():
        dp.create_dataset(p, data=arr)
    for p, arr in vectors.items():
        dp.create_dataset(p, data=arr)
    for p, v in flag_values.items():
        dp.create_dataset(p, data=np.bool_(v))

print(f"V1 nodes written to {v1_nodes_path} ({v1_nodes_path.stat().st_size / 1e6:.1f} MB)")
print(f"  model_template: {NEST_MODEL}")
print(f"  scalar params (per-cell, shape {(n_v1,)}): {SCALAR_PARAMS}")
print(f"  vector params (per-cell): {VECTOR_PARAMS}")
print(f"  flag params (scalar): {flag_values}")

V1 nodes written to ../../../obi-output/circuit/nest_core_nll_7/v1_nodes.h5 (27.9 MB)
  model_template: glif_psc_double_alpha
  scalar params (per-cell, shape (66952,)): ['V_m', 'V_th', 'g', 'E_L', 'C_m', 't_ref', 'V_reset']
  vector params (per-cell): {'asc_init': 2, 'asc_decay': 2, 'asc_amps': 2, 'tau_syn_fast': 10, 'tau_syn_slow': 10, 'amp_slow': 10}
  flag params (scalar): {'spike_dependent_threshold': False, 'after_spike_currents': True, 'adapting_threshold': False}


### Write LGN and BKG virtual node populations

These populations are `spike_generator`s at simulation time, so they carry
no `model_template` / `dynamics_params`. We still rewrite them into the
output directory so the new circuit config references a self-contained
set of HDF5 files (rather than pointing back into `core_nll_7/network/`).
Per-cell metadata from the original files (filter parameters for LGN,
positions for BKG) is preserved under `nodes/<pop>/0/` for downstream
analysis code that expects it.

In [4]:
def copy_virtual_population(src_path: Path, dst_path: Path, pop_name: str) -> int:
    """Rewrite a virtual SONATA node population into *dst_path*.

    Copies `node_id`, `node_type_id`, `node_group_id`, `node_group_index`
    and every per-cell dataset under `0/` from the source file, using
    `uint64` dtypes to match the `fly_nodes.h5` convention.
    """
    with h5py.File(src_path, "r") as src, h5py.File(dst_path, "w") as dst:
        src_pop = src[f"nodes/{pop_name}"]
        n = int(src_pop["node_id"].shape[0])
        dst_pop = dst.create_group(f"nodes/{pop_name}")
        dst_pop.create_dataset("node_id", data=src_pop["node_id"][:].astype(np.uint64))
        dst_pop.create_dataset("node_type_id", data=src_pop["node_type_id"][:].astype(np.uint64))
        dst_pop.create_dataset("node_group_id", data=np.zeros(n, dtype=np.uint64))
        dst_pop.create_dataset("node_group_index", data=np.arange(n, dtype=np.uint64))
        dst_grp = dst_pop.create_group("0")
        if "0" in src_pop:
            for name, ds in src_pop["0"].items():
                dst_grp.create_dataset(name, data=ds[()])
    return n


lgn_nodes_path = OUTPUT_DIR / "lgn_nodes.h5"
n_lgn = copy_virtual_population(NETWORK_DIR / "lgn_nodes.h5", lgn_nodes_path, "lgn")
print(f"LGN nodes written to {lgn_nodes_path} ({n_lgn:,} virtual neurons)")

bkg_nodes_path = OUTPUT_DIR / "bkg_nodes.h5"
n_bkg = copy_virtual_population(NETWORK_DIR / "bkg_nodes.h5", bkg_nodes_path, "bkg")
print(f"BKG nodes written to {bkg_nodes_path} ({n_bkg:,} virtual neurons)")

LGN nodes written to ../../../obi-output/circuit/nest_core_nll_7/lgn_nodes.h5 (17,400 virtual neurons)
BKG nodes written to ../../../obi-output/circuit/nest_core_nll_7/bkg_nodes.h5 (100 virtual neurons)


### Write `node_sets.json` and `circuit_config.json`

The circuit config points at the new node HDF5 files we just wrote. The
edge files stay where they are in `core_nll_7/network/` (this notebook only
rewrites nodes).

In [5]:
node_sets = {
    "All": {"population": "v1"},
    "v1": {"population": "v1"},
    "lgn": {"population": "lgn"},
    "bkg": {"population": "bkg"},
}
node_sets_path = OUTPUT_DIR / "node_sets.json"
with node_sets_path.open("w", encoding="utf-8") as f:
    json.dump(node_sets, f, indent=2)

nd = str(NETWORK_DIR)
circuit_config = {
    "manifest": {"$BASE_DIR": "."},
    "node_sets_file": str(node_sets_path.resolve().absolute()),
    "networks": {
        "nodes": [
            {
                "nodes_file": str(v1_nodes_path.resolve().absolute()),
                "populations": {"v1": {"type": "point_process"}},
            },
            {
                "nodes_file": str(lgn_nodes_path.resolve().absolute()),
                "populations": {"lgn": {"type": "virtual"}},
            },
            {
                "nodes_file": str(bkg_nodes_path.resolve().absolute()),
                "populations": {"bkg": {"type": "virtual"}},
            },
        ],
        "edges": [
            {
                "edges_file": f"{nd}/v1_v1_edges_bio_trained.h5",
                "populations": {"v1_to_v1": {"type": "chemical"}},
            },
            {
                "edges_file": f"{nd}/lgn_v1_edges.h5",
                "populations": {"lgn_to_v1": {"type": "chemical"}},
            },
            {
                "edges_file": f"{nd}/bkg_v1_edges_bio_trained.h5",
                "populations": {"bkg_to_v1": {"type": "chemical"}},
            },
        ],
    },
}

circuit_config_path = OUTPUT_DIR / "circuit_config.json"
with circuit_config_path.open("w", encoding="utf-8") as f:
    json.dump(circuit_config, f, indent=2)

print(f"Circuit config written to {circuit_config_path}")
print(f"Node sets written to {node_sets_path}")

Circuit config written to ../../../obi-output/circuit/nest_core_nll_7/circuit_config.json
Node sets written to ../../../obi-output/circuit/nest_core_nll_7/node_sets.json


### Verify

In [6]:
with circuit_config_path.open() as f:
    written = json.load(f)
print(f"Node populations: {[next(iter(n['populations'])) for n in written['networks']['nodes']]}")
print(f"Edge populations: {[next(iter(e['populations'])) for e in written['networks']['edges']]}")

with h5py.File(v1_nodes_path, "r") as f:
    grp = f["nodes/v1/0"]
    n = int(f["nodes/v1/node_type_id"].shape[0])
    model = grp["model_template"][()].decode()
    dp = grp["dynamics_params"]
    print(f"\nV1 population: {n:,} neurons, model_template='{model}'")
    for name in sorted(dp):
        ds = dp[name]
        print(f"  dynamics_params/{name}: shape={ds.shape} dtype={ds.dtype}")

for nodes_path, pop in [(lgn_nodes_path, "lgn"), (bkg_nodes_path, "bkg")]:
    with h5py.File(nodes_path, "r") as f:
        n = int(f[f"nodes/{pop}/node_id"].shape[0])
        extra = sorted(f[f"nodes/{pop}/0"]) if "0" in f[f"nodes/{pop}"] else []
        print(f"\n{pop.upper()} (virtual): {n:,} neurons, 0/ datasets: {extra}")

Node populations: ['v1', 'lgn', 'bkg']
Edge populations: ['v1_to_v1', 'lgn_to_v1', 'bkg_to_v1']

V1 population: 66,952 neurons, model_template='glif_psc_double_alpha'
  dynamics_params/C_m: shape=(66952,) dtype=float64
  dynamics_params/E_L: shape=(66952,) dtype=float64
  dynamics_params/V_m: shape=(66952,) dtype=float64
  dynamics_params/V_reset: shape=(66952,) dtype=float64
  dynamics_params/V_th: shape=(66952,) dtype=float64
  dynamics_params/adapting_threshold: shape=() dtype=bool
  dynamics_params/after_spike_currents: shape=() dtype=bool
  dynamics_params/amp_slow: shape=(66952, 10) dtype=float64
  dynamics_params/asc_amps: shape=(66952, 2) dtype=float64
  dynamics_params/asc_decay: shape=(66952, 2) dtype=float64
  dynamics_params/asc_init: shape=(66952, 2) dtype=float64
  dynamics_params/g: shape=(66952,) dtype=float64
  dynamics_params/spike_dependent_threshold: shape=() dtype=bool
  dynamics_params/t_ref: shape=(66952,) dtype=float64
  dynamics_params/tau_syn_fast: shape=(6695